<a href="https://colab.research.google.com/github/makuhinkonstantin545-collab/ai-security-labs/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_1_%D0%91%D0%B0%D0%B5%D0%B9%D1%81%D0%BE%D0%B2%D1%81%D0%BA%D0%B8%D0%B9_%D1%81%D0%BF%D0%B0%D0%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Привет, Python!")

Привет, Python!


In [3]:
messages = [
    "Поздравляем! Вы выиграли 100000 рублей!",
    "Срочно перейдите по ссылке и получите приз",
    "Вы стали счастливым обладателем подарка",
    "Получите бесплатный бонус прямо сейчас",
    "Ваша карта заблокирована, подтвердите данные",

    "Привет! Как дела?",
    "Увидимся завтра в университете",
    "Не забудь отправить мне лабораторную",
    "Сегодня вечером посмотрим фильм?",
    "Когда у тебя следующая пара?",

    "Вы выиграли новый смартфон!",
    "Нажмите на ссылку для получения денег",
    "Специальное предложение только сегодня",
    "Получите скидку 90 процентов прямо сейчас",
    "Ваш приз ожидает вас",

    "Я уже приехал домой",
    "Можешь прислать конспект?",
    "Завтра встречаемся в 10 утра",
    "Спасибо за помощь!",
    "Давай обсудим это завтра",
    "Я отправил тебе файл",
    "До встречи!"
]

labels = [
    "spam", "spam", "spam", "spam", "spam",
    "ham", "ham", "ham", "ham", "ham",
    "spam", "spam", "spam", "spam", "spam",
    "ham", "ham", "ham", "ham", "ham", "ham", "ham"
]

print("Количество сообщений:", len(messages))
print("Количество меток:", len(labels))

Количество сообщений: 22
Количество меток: 22


In [4]:
import pandas as pd

data = pd.DataFrame({
    "text": messages,
    "label": labels
})

data

,text,label
0,Поздравляем! Вы выиграли 100000 рублей!,spam
1,Срочно перейдите по ссылке и получите приз,spam
2,Вы стали счастливым обладателем подарка,spam
3,Получите бесплатный бонус прямо сейчас,spam
4,"Ваша карта заблокирована, подтвердите данные",spam
5,Привет! Как дела?,ham
6,Увидимся завтра в университете,ham
7,Не забудь отправить мне лабораторную,ham
8,Сегодня вечером посмотрим фильм?,ham
9,Когда у тебя следующая пара?,ham


In [6]:
import re

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
test_message = "ПОЛУЧИТЕ!!! 100000 рублей прямо сейчас!!!"

print("До очистки:")
print(test_message)

print("\nПосле очистки:")
print(preprocess_text(test_message))

До очистки:
ПОЛУЧИТЕ!!! 100000 рублей прямо сейчас!!!

После очистки:
получите рублей прямо сейчас


In [8]:
data["clean_text"] = data["text"].apply(preprocess_text)

data[["text", "clean_text", "label"]]

,text,clean_text,label
0,Поздравляем! Вы выиграли 100000 рублей!,поздравляем вы выиграли рублей,spam
1,Срочно перейдите по ссылке и получите приз,срочно перейдите по ссылке и получите приз,spam
2,Вы стали счастливым обладателем подарка,вы стали счастливым обладателем подарка,spam
3,Получите бесплатный бонус прямо сейчас,получите бесплатный бонус прямо сейчас,spam
4,"Ваша карта заблокирована, подтвердите данные",ваша карта заблокирована подтвердите данные,spam
5,Привет! Как дела?,привет как дела,ham
6,Увидимся завтра в университете,увидимся завтра в университете,ham
7,Не забудь отправить мне лабораторную,не забудь отправить мне лабораторную,ham
8,Сегодня вечером посмотрим фильм?,сегодня вечером посмотрим фильм,ham
9,Когда у тебя следующая пара?,когда у тебя следующая пара,ham


In [9]:
from sklearn.model_selection import train_test_split

X = data["clean_text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Обучающих сообщений:", len(X_train))
print("Тестовых сообщений:", len(X_test))

Обучающих сообщений: 15
Тестовых сообщений: 7


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    token_pattern=r'(?u)\b\w+\b',
    min_df=1,
    max_df=0.8,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Размер обучающей матрицы:", X_train_tfidf.shape)
print("Размер тестовой матрицы:", X_test_tfidf.shape)

Размер обучающей матрицы: (15, 105)
Размер тестовой матрицы: (7, 105)


In [11]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB(alpha=0.5)

model.fit(X_train_tfidf, y_train)

print("Модель обучена!")

Модель обучена!


In [12]:
y_pred = model.predict(X_test_tfidf)

print("Предсказания модели:")
print(y_pred)

print("\nПравильные ответы:")
print(y_test.to_numpy())

Предсказания модели:
['ham' 'ham' 'spam' 'ham' 'ham' 'ham' 'spam']

Правильные ответы:
['ham' 'ham' 'spam' 'ham' 'ham' 'spam' 'spam']


In [13]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8571428571428571

Classification Report:
              precision    recall  f1-score   support

         ham       0.80      1.00      0.89         4
        spam       1.00      0.67      0.80         3

    accuracy                           0.86         7
   macro avg       0.90      0.83      0.84         7
weighted avg       0.89      0.86      0.85         7



In [14]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Матрица ошибок:")
print(cm)

Матрица ошибок:
[[4 0]
 [1 2]]


In [15]:
new_messages = [
    "Вы выиграли миллион рублей! Получите свой приз прямо сейчас!",
    "Привет, не забудь отправить мне отчет вечером",
    "Вам доступно специальное предложение"
]

new_messages_clean = [preprocess_text(msg) for msg in new_messages]

new_messages_tfidf = vectorizer.transform(new_messages_clean)

new_predictions = model.predict(new_messages_tfidf)

for message, prediction in zip(new_messages, new_predictions):
    print("Сообщение:", message)
    print("Предсказание:", prediction)
    print()

Сообщение: Вы выиграли миллион рублей! Получите свой приз прямо сейчас!
Предсказание: spam

Сообщение: Привет, не забудь отправить мне отчет вечером
Предсказание: ham

Сообщение: Вам доступно специальное предложение
Предсказание: ham



In [16]:
new_probabilities = model.predict_proba(new_messages_tfidf)

for message, prediction, probabilities in zip(
    new_messages,
    new_predictions,
    new_probabilities
):
    print("Сообщение:", message)
    print("Предсказание:", prediction)
    print(
        f"Вероятность ham: {probabilities[0]:.2f}"
    )
    print(
        f"Вероятность spam: {probabilities[1]:.2f}"
    )
    print()

Сообщение: Вы выиграли миллион рублей! Получите свой приз прямо сейчас!
Предсказание: spam
Вероятность ham: 0.11
Вероятность spam: 0.89

Сообщение: Привет, не забудь отправить мне отчет вечером
Предсказание: ham
Вероятность ham: 0.84
Вероятность spam: 0.16

Сообщение: Вам доступно специальное предложение
Предсказание: ham
Вероятность ham: 0.53
Вероятность spam: 0.47

